# 1412. Find the Quiet Students in All Exams

## Question
We need to find students who are **quiet** in all exams.  
A "quiet" student is defined as:
- Someone who has taken at least one exam.
- Never scored the **highest** or **lowest** in any exam.

Return the list of such students ordered by `student_id`.

---

## Schema

### Table: Student
| Column Name  | Type    | Description                  |
|--------------|---------|------------------------------|
| student_id   | INT     | Primary key                  |
| student_name | STRING  | Name of the student          |

---

### Table: Exam
| Column Name | Type    | Description                                |
|-------------|---------|--------------------------------------------|
| exam_id     | INT     | Exam identifier                            |
| student_id  | INT     | Foreign key referencing Student            |
| score       | INT     | Score obtained by the student in the exam  |

**Primary Key:** (exam_id, student_id)

---

## Sample Data

### Student
| student_id | student_name |
|------------|--------------|
| 1          | Daniel       |
| 2          | Jade         |
| 3          | Stella       |
| 4          | Jonathan     |
| 5          | Will         |

### Exam
| exam_id | student_id | score |
|---------|------------|-------|
| 10      | 1          | 70    |
| 10      | 2          | 80    |
| 10      | 3          | 90    |
| 20      | 1          | 80    |
| 30      | 1          | 70    |
| 30      | 3          | 80    |
| 30      | 4          | 90    |
| 40      | 1          | 60    |
| 40      | 2          | 70    |
| 40      | 4          | 80    |

---

## PySpark Code to Create Schema, Data, and Temp Views

```python
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Schema for Student
student_schema = StructType([
    StructField("student_id", IntegerType(), False),
    StructField("student_name", StringType(), False)
])

# Schema for Exam
exam_schema = StructType([
    StructField("exam_id", IntegerType(), False),
    StructField("student_id", IntegerType(), False),
    StructField("score", IntegerType(), False)
])

# Data for Student
student_data = [
    (1, "Daniel"),
    (2, "Jade"),
    (3, "Stella"),
    (4, "Jonathan"),
    (5, "Will")
]

# Data for Exam
exam_data = [
    (10, 1, 70),
    (10, 2, 80),
    (10, 3, 90),
    (20, 1, 80),
    (30, 1, 70),
    (30, 3, 80),
    (30, 4, 90),
    (40, 1, 60),
    (40, 2, 70),
    (40, 4, 80)
]

# Create DataFrames
student_df = spark.createDataFrame(student_data, student_schema)
exam_df = spark.createDataFrame(exam_data, exam_schema)

# Register Temp Views
student_df.createOrReplaceTempView("Student")
exam_df.createOrReplaceTempView("Exam")

# Quick check
student_df.show()
exam_df.show()


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Schema for Student
student_schema = StructType([
    StructField("student_id", IntegerType(), False),
    StructField("student_name", StringType(), False)
])

# Schema for Exam
exam_schema = StructType([
    StructField("exam_id", IntegerType(), False),
    StructField("student_id", IntegerType(), False),
    StructField("score", IntegerType(), False)
])

# Data for Student
student_data = [
    (1, "Daniel"),
    (2, "Jade"),
    (3, "Stella"),
    (4, "Jonathan"),
    (5, "Will")
]

# Data for Exam
exam_data = [
    (10, 1, 70),
    (10, 2, 80),
    (10, 3, 90),
    (20, 1, 80),
    (30, 1, 70),
    (30, 3, 80),
    (30, 4, 90),
    (40, 1, 60),
    (40, 2, 70),
    (40, 4, 80)
]

# Create DataFrames
student_df = spark.createDataFrame(student_data, student_schema)
exam_df = spark.createDataFrame(exam_data, exam_schema)

# Register Temp Views
student_df.createOrReplaceTempView("Student")
exam_df.createOrReplaceTempView("Exam")

# Quick check
student_df.show()
exam_df.show()

In [0]:
%sql
with cte as (
  Select
    exam_id ,
    student_id  , 
    score   ,
    min(score)over(partition by exam_id) as min_score,
    max(score)over(partition by exam_id) as max_score
from  exam 
) 
, cte_2 as (
select *
 from cte 
    where (score = min_score or score = max_score) 
    order by exam_id asc
)
select * from student where student_id not in (select distinct student_id from cte_2)
and student_id  in (select distinct student_id from cte)

In [0]:
%sql
with cte as (
  Select
    exam_id ,
    student_id  , 
    score   ,
    min(score)over(partition by exam_id) as min_score,
    max(score)over(partition by exam_id) as max_score
from  exam 
) 
, cte_2 as (
select *
 from cte 
    where (score = min_score or score = max_score) 
    order by exam_id asc
) 
, cte3 as (
select distinct student_id  from cte  where student_id not in (select distinct student_id from cte_2)
)
select s.* from Student  s inner join cte3 on s.student_id = cte3.student_id

# Thought Process: Quiet Students in Exams

## Approach
- First, I calculated the **minimum** and **maximum** score for each `exam_id`.  
- Next, I identified the students who scored either the minimum or maximum in those exams.  
- Once I had this set, I could then determine which students were **not** among the min/max scorers.

## Challenge
- Some students never appeared for any exam, which means they would incorrectly show up as "quiet" students.  
- To handle this, I filtered only from the CTE that was built using the **Exam** table, ensuring I only considered students who actually took part in at least one exam.

## Refinement
- At the end, I excluded students who never appeared in any exam.  
- Initially, I used **two CTEs** and **two subqueries** outside the CTEs.  
- Since subqueries are less efficient and harder to read, I added one more CTE and replaced the subqueries with a **JOIN**, making the query cleaner and more consistent.
